In [ ]:
import ast
import sys
sys.path.insert(0, "../../wildlife-datasets")

import matplotlib.pyplot as plt
import numpy as np

from scipy.optimize import curve_fit
from wildlife_datasets.datasets import TurtlesOfSMSRC

In [ ]:
def negative_exponential(x, a, b):
    return a * (1 - np.exp(-b * x))

def michaelis_menten(x, a, b):
    return (a * x) / (b + x)

In [ ]:
with open("hidden/matched_encounters.txt") as f:
    replace_identity = ast.literal_eval(f.read())

root = '/data/wildlife_datasets/data/TurtlesOfSMSRC'
dataset = TurtlesOfSMSRC(root, replace_identity=replace_identity)

In [ ]:
len(dataset), dataset.df.encounter_id.nunique(), dataset.df.identity.nunique()

In [ ]:
encounter_counts = [(x, df["encounter_id"].nunique()) for x, df in dataset.metadata.groupby("identity")]
encounter_counts = sorted(encounter_counts, key = lambda x: x[1], reverse = True)
encounter_counts = np.array(encounter_counts)

plt.plot(encounter_counts[:, 1])
plt.xlabel("Individual turtles")
plt.ylabel("Number of encounters")
plt.grid()

In [ ]:
n_super = 3
for i, (identity, _) in enumerate(encounter_counts[:n_super]):
    mask = dataset.metadata["identity"] == identity
    metadata_encounters = dataset.metadata.loc[mask, "encounter_id"]
    print(f"Super resident {i+1} observed {metadata_encounters.nunique()} times")
    print(metadata_encounters.unique())

In [ ]:
n_individuals = []
encounters = np.sort(dataset.metadata["encounter_id"].unique())
for encounter in encounters:
    mask = dataset.metadata["encounter_id"] <= encounter
    n_individuals.append(dataset.metadata.loc[mask, "identity"].nunique())

xs = np.arange(1, len(n_individuals)+1)
x_line = np.linspace(1, 5*max(xs), 200)

model = negative_exponential
fit_results, _ = curve_fit(model, xs, n_individuals, maxfev=5000)

plt.plot(x_line, model(x_line, *fit_results), label="Population: estimate")
plt.plot(xs, n_individuals, label="Population: data")
plt.plot(xs, xs, label="Upper bound (identity mapping)", color="black")

plt.title("Predicted individuals (negative exponential model)")
plt.xlabel("Number of encounters")
plt.ylabel("Number of individuals")
plt.legend()